In [1]:
import sys
from pathlib import Path

repo_root = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN')
sys.path.insert(0, str(repo_root))


### 1. Configuração do Ambiente

In [2]:
#!pip install "cognite-sdk[pandas]" matplotlib seaborn tensorflow plotly -q

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.stattools import acf
from industrial_ts.dataloader import DataLoader
import json
from getpass import getpass
#import tensorflow as tf
#from tensorflow import keras

print("Bibliotecas importadas com sucesso!")






Updated: UnsupervisedStateSegmenter now supports multi-channel (multivariate) series.

Bibliotecas importadas com sucesso!


/home/ferna/fe/lib/python3.12/site-packages/ipykernel/ipkernel.py:772: UserWarning: You are using version='7.83.1' of the SDK, however version='7.91.2' is available. To suppress this warning, either upgrade or do the following:
>>> from cognite.client.config import global_config
>>> global_config.disable_pypi_version_check = True
  _threading_Thread_run(self)


In [3]:
import debugpy
debugpy.listen(('0.0.0.0', 5678))






('0.0.0.0', 5678)

In [4]:
os.environ['COGNITE_CLIENT_SECRET'] = getpass("Enter COGNITE_CLIENT")






### 2. Ativar o DataLoader

In [5]:
import importlib
import sys
importlib.reload(sys.modules['industrial_ts.dataloader'])
from industrial_ts.dataloader import DataLoader
dl = DataLoader()
dl.add_segments(segments=3, window=10, step=10, series=[
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActShaft Power',
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActPress Ratio'  
], path="segmenter_model_10.pkl"
)
#dl.segmenter.save("segmenter_model_10.pkl")








Buscando dados para as 12 séries temporais encontradas.


### 10 Pós-processamento

In [6]:
for i in [2]:
    dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2







/tmp/ipykernel_121354/4230046523.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2


In [7]:
dl.add_time_to_change_state_timestamp()






## Experimento 1: seqKAN  grid 20 clamp -10/10  Adam 

In [18]:
from industrial_ts.seqKAN import TSDF_seqKAN







In [19]:
kan_params = {
    'hidden': {'grid_eps': 1.0, 'grid': 20, 'k': 3, 'grid_range': (-10, 10)},
        'sparse_init': True,
    'output': {'grid_eps': 1.0, 'grid': 20, 'k': 3, 'grid_range': (-10, 10)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


### Atualização da dinâmica

Nesta versão, a atualização é:
`h_t = h_{t-1} + m * s1(x_t) + (1 - m) * z_{t-1}`
com `z_t` calculado primeiro em paralelo.


In [20]:
import time
train_start = time.time()







In [21]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='adam',
        optimizer_params={'lr': 2e-4},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-10.0,
        x_max=10.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 3.06%
>% |x| > 3 : 1.21%
>% |x| > 4 : 0.81%
min: -10.0 max: 9.25381088256836
p1/p50/p99: [-2.94778953  0.          2.13391382]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -10.0
worst_max_feature: PH (CBM) 1st Stage ExpShaft Power 9.25381088256836
Epoch 1/150 | Train(sampled) L1:1.131417 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:1.093178 ± 0.531531 | Val micro:0.667735 ± 0.046190 
          >> Test macro:1.039324 ± 0.545497 | micro:0.602430 ± 0.029618
>% |x| > 2 : 2.95%
>% |x| > 3 : 1.09%
>% |x| > 4 : 0.70%
min: -10.0 max: 7.229738235473633
p1/p50/p99: [-2.7326973  0.         2.1199627]
worst_min_feature: PH (CBM) 1st Stage ExpCompr Poly Eff -10.0
worst_max_feature: PH (CBM) 1st Stage ExpShaft Power 7.229738235473633
Epoch 2/150 | Train(sampled) L1:1.161011 L2:0.000000 L3:0.000000  L4:0.000000 L5:

In [22]:

##### Tempo de treinamento#######

train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")

Tempo total de treino: 11.57 min (694.0 s)
Melhor época: 147 | melhor val: 0.274725


## Experimento 2 : grid 40 -6/6 RAdam


In [27]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-6, 6)},
        'sparse_init': True,
    'output': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-6, 6)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [28]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-6.0,
        x_max=6.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


##### Tempo de treinamento#######

train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")

/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 1.23%
>% |x| > 4 : 0.81%
min: -6.0 max: 6.0
p1/p50/p99: [-3.0347794  -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ActShaft Power -6.0
worst_max_feature: PH (CBM) 1st Stage ActShaft Power 6.0
Epoch 1/150 | Train(sampled) L1:0.991741 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.981702 ± 0.480772 | Val micro:0.596887 ± 0.022636 
          >> Test macro:0.966950 ± 0.491780 | micro:0.573079 ± 0.018151
>% |x| > 2 : 2.96%
>% |x| > 3 : 1.15%
>% |x| > 4 : 0.71%
min: -6.0 max: 6.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stg ActCompr Poly Head -6.0
worst_max_feature: PH (CBM) 1st Stage ActShaft Power 6.0
Epoch 2/150 | Train(sampled) L1:1.008166 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.960813 ± 0.465267 | V

## Experimento 3 : grid 80 -6/6 Rdam


In [29]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.02, 'grid': 80, 'k': 3, 'grid_range': (-6, 6)},
        'sparse_init': True,
    'output': {'grid_eps': 0.02, 'grid': 80, 'k': 3, 'grid_range': (-6, 6)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [30]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-6.0,
        x_max=6.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 3.14%
>% |x| > 3 : 1.31%
>% |x| > 4 : 0.89%
min: -6.0 max: 6.0
p1/p50/p99: [-3.18839561  0.          2.12642989]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -6.0
worst_max_feature: PH (CBM) 1st Stage ActShaft Power 6.0
Epoch 1/150 | Train(sampled) L1:0.995118 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.974517 ± 0.467560 | Val micro:0.600277 ± 0.022704 
          >> Test macro:0.959484 ± 0.479882 | micro:0.575142 ± 0.018135
>% |x| > 2 : 2.78%
>% |x| > 3 : 1.10%
>% |x| > 4 : 0.76%
min: -6.0 max: 6.0
p1/p50/p99: [-2.74456794  0.          2.05935931]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -6.0
worst_max_feature: PH (CBM) 1st Stage ActShaft Power 6.0
Epoch 2/150 | Train(sampled) L1:0.994901 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.957945 ± 0.456384 | Val

## Experimento 4: grid 40 -8/8 RAdam


In [31]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-8, 8)},
        'sparse_init': True,
    'output': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-8, 8)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [32]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-8.0,
        x_max=8.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 1.23%
>% |x| > 4 : 0.81%
min: -8.0 max: 8.0
p1/p50/p99: [-3.0347794  -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ActShaft Power -8.0
worst_max_feature: PH (CBM) 1st Stage ActShaft Power 8.0
Epoch 1/150 | Train(sampled) L1:1.062268 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:1.051012 ± 0.525534 | Val micro:0.630369 ± 0.032889 
          >> Test macro:1.016739 ± 0.535248 | micro:0.588053 ± 0.023553
>% |x| > 2 : 2.96%
>% |x| > 3 : 1.15%
>% |x| > 4 : 0.71%
min: -8.0 max: 8.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stage ExpCompr Poly Eff -8.0
worst_max_feature: PH (CBM) 1st Stage ActShaft Power 8.0
Epoch 2/150 | Train(sampled) L1:1.091808 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:1.032998 ± 0.509261 | 

## Experimento: grid 40 -4/4RAdam


In [34]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-4, 4)},
        'sparse_init': True,
    'output': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-4, 4)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [35]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-4.0,
        x_max=4.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 1.23%
>% |x| > 4 : 0.00%
min: -4.0 max: 4.0
p1/p50/p99: [-3.0347794  -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -4.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 4.0
Epoch 1/150 | Train(sampled) L1:0.890167 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.883761 ± 0.411054 | Val micro:0.554749 ± 0.013551 
          >> Test macro:0.875374 ± 0.409830 | micro:0.547136 ± 0.012196
>% |x| > 2 : 2.96%
>% |x| > 3 : 1.15%
>% |x| > 4 : 0.00%
min: -4.0 max: 4.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stg ActCompr Poly Head -4.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 4.0
Epoch 2/150 | Train(sampled) L1:0.889282 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.861147 ± 0.396850 | Val

## Experimento: grid 40 -3/3 RAdam  

In [14]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [15]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.         -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.827192 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.823579 ± 0.364544 | Val micro:0.531794 ± 0.010016 
          >> Test macro:0.815043 ± 0.355593 | micro:0.530245 ± 0.009450
>% |x| > 2 : 2.96%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.819718 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.801801 ± 0.351227 | Val m

In [16]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 14.17 min (850.2 s)
Melhor época: 76 | melhor val: 0.238297


## Experimento grid 40 -3/3 RAdam grideps 0.2

In [8]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [9]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.         -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.827192 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.823579 ± 0.364544 | Val micro:0.531794 ± 0.010016 
          >> Test macro:0.815043 ± 0.355593 | micro:0.530245 ± 0.009450
>% |x| > 2 : 2.96%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.819718 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.801801 ± 0.351227 | Val m

# ## Experimento grid 40 -3/3 RAdam grideps 0.5

In [10]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.5, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.5, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [11]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.         -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.827192 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.823579 ± 0.364544 | Val micro:0.531794 ± 0.010016 
          >> Test macro:0.815043 ± 0.355593 | micro:0.530245 ± 0.009450
>% |x| > 2 : 2.96%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.819718 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.801801 ± 0.351227 | Val m

# ## Experimento grid 40 -3/3 RAdam grideps 1

In [12]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.5, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.5, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [13]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.         -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.827192 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.823579 ± 0.364544 | Val micro:0.531794 ± 0.010016 
          >> Test macro:0.815043 ± 0.355593 | micro:0.530245 ± 0.009450
>% |x| > 2 : 2.96%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.819718 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.801801 ± 0.351227 | Val m

# ## Experimento grid 60 -3/3 RAdam gridep0,2

In [17]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.02, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [18]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.89%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.         -0.          2.07787437]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.827192 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.823579 ± 0.364544 | Val micro:0.531794 ± 0.010016 
          >> Test macro:0.815043 ± 0.355593 | micro:0.530245 ± 0.009450
>% |x| > 2 : 2.96%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.78219082  0.          2.11758576]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.819718 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.801801 ± 0.351227 | Val m

In [19]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 12.38 min (742.6 s)
Melhor época: 73 | melhor val: 0.238055


# Experimento com hidden=11 - grid 40 -3/3 Radam eps 0,2

In [8]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft',
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [9]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 3.17%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.          0.          2.09904362]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.842637 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.863902 ± 0.340154 | Val micro:0.591639 ± 0.010361 
          >> Test macro:0.851278 ± 0.332238 | micro:0.585185 ± 0.009672
>% |x| > 2 : 2.93%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.93194993 -0.          2.08784839]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.848755 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.853368 ± 0.340553 | Val m

In [10]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 10.75 min (645.2 s)
Melhor época: 113 | melhor val: 0.239179


# Experimento com hidden=11 - grid 40 -3/3 Radam eps 0,2  topK32

In [8]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft', ## ou hard
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


In [9]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 3.17%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.          0.          2.09904362]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.842636 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.864065 ± 0.340189 | Val micro:0.591775 ± 0.010361 
          >> Test macro:0.851438 ± 0.332261 | micro:0.585327 ± 0.009672
>% |x| > 2 : 2.93%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.93194993 -0.          2.08784839]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.849565 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.855842 ± 0.342068 | Val m

In [10]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 18.68 min (1120.8 s)
Melhor época: 140 | melhor val: 0.199045


# Experimento com hidden=11 - grid 40 -3/3 Radam eps 0,2  topK11

In [11]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 40, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft', ## ou hard
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [12]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 3.17%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.          0.          2.09904362]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.842636 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.864065 ± 0.340189 | Val micro:0.591775 ± 0.010361 
          >> Test macro:0.851438 ± 0.332261 | micro:0.585327 ± 0.009672
>% |x| > 2 : 2.93%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.93194993 -0.          2.08784839]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.849565 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.855842 ± 0.342068 | Val m

In [13]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 16.11 min (966.6 s)
Melhor época: 140 | melhor val: 0.199050


# Experimento com hidden=24 - grid 8 -3/3 Radam eps 0,2  topK 4

In [14]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft', ## ou hard
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [15]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=11,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:731: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 2.93%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.6803442   0.          2.14367849]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.856234 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.862964 ± 0.315704 | Val micro:0.610272 ± 0.009871 
          >> Test macro:0.857378 ± 0.312913 | micro:0.606763 ± 0.009324
>% |x| > 2 : 2.86%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-2.84155745  0.          2.09803688]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.855249 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.855635 ± 0.314019 | Val m

In [16]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 16.75 min (1004.9 s)
Melhor época: 149 | melhor val: 0.201651


## Experimento Baseline GRU

In [39]:
from pathlib import Path
from industrial_ts.gru import TSDF_GRU
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_gru(cfg):
    tp = cfg.get('train_params', {})
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"gru_{clamp}_{opt}_lr{lr}")

run_name_gru = _make_run_name_gru(run_config_gru)
out_json_gru = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_reuslts') / f"final_rebuild_{run_name_gru}.json")

# Se run_config nao existir (celula anterior nao executada), cria configuracao base
if 'run_config' not in globals():
    batch_size = 512
    run_config = dict(
        model='TSDF_seqKAN',
        in_channels=11,
        hidden_dim=11*32,
        cost_columns=[
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActShaft Power',
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActPress Ratio'
        ],
        lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        sigma_temp=0.6,
        log_likelihood=False,
        use_layernorm=False,
        direct_x=True,
        train_params=dict(
            batch_size=batch_size,
            window_size=10,
            window_step=10,
            epochs=150,
            validate=True,
            patience=20,
            kl_warmup_epochs=20,
            kl_start=0.01,
            rebuild=True,
            reconstruction_test=False,
            warmup_steps=0,
            min_lr_factor=1.0,
            lr_scheduler='None',
            plateau_monitor='val_micro',
            plateau_factor=0.5,
            plateau_patience=8,
            plateau_min_lr=1e-6,
            spline_soft_freeze_epoch=None,
            spline_soft_freeze_factor=0.02,
            optimizer_name='radam',
            optimizer_params={'lr': 2e-4},
            grad_clip_max_norm=0.3,
            debug_batch_stats=True,
            x_min=None,
            x_max=None,
            save_best_ckpt=True,
        ),
    )

run_config_gru = dict(
    model='TSDF_GRU',
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    train_params=run_config['train_params'],
)

model_gru = TSDF_GRU(
    in_channels=run_config_gru['in_channels'],
    hidden_dim=run_config_gru['hidden_dim'],
    cost_columns=run_config_gru['cost_columns'],
    lam=run_config_gru['lam'],
    sigma_temp=run_config_gru['sigma_temp'],
    log_likelihood=run_config_gru['log_likelihood'],
    use_layernorm=run_config_gru['use_layernorm'],
)

run_config_gru['train_params']['x_min'] = None

# Garante sem clamp para GRU

import time
train_start_gru = time.time()
res_gru = model_gru.train_cognite(

    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_gru['train_params']['batch_size'],
    window_size=run_config_gru['train_params']['window_size'],
    window_step=run_config_gru['train_params']['window_step'],
    epochs=run_config_gru['train_params']['epochs'],
    validate=run_config_gru['train_params']['validate'],
    patience=run_config_gru['train_params']['patience'],
    kl_warmup_epochs=run_config_gru['train_params']['kl_warmup_epochs'],
    kl_start=run_config_gru['train_params']['kl_start'],
    rebuild=run_config_gru['train_params']['rebuild'],
    reconstruction_test=run_config_gru['train_params']['reconstruction_test'],
    warmup_steps=run_config_gru['train_params']['warmup_steps'],
    min_lr_factor=run_config_gru['train_params']['min_lr_factor'],
    lr_scheduler=run_config_gru['train_params']['lr_scheduler'],
    plateau_monitor=run_config_gru['train_params']['plateau_monitor'],
    plateau_factor=run_config_gru['train_params']['plateau_factor'],
    plateau_patience=run_config_gru['train_params']['plateau_patience'],
    plateau_min_lr=run_config_gru['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config_gru['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config_gru['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config_gru['train_params']['save_best_ckpt'],
    optimizer_name=run_config_gru['train_params']['optimizer_name'],
    optimizer_params=run_config_gru['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config_gru['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config_gru['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config_gru['train_params']['x_min'],
    x_max=run_config_gru['train_params']['x_max'],
)
train_end_gru = time.time()
train_seconds_gru = train_end_gru - train_start_gru
train_minutes_gru = train_seconds_gru / 60.0


res_gru = [r for r in res_gru if r is not None]
model_gru.save('gru_last_l1.pt')
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_gru)

# Best epoch by test micro
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_gru)

summary = _extract_summary(res_gru)

with open(out_json_gru, 'w') as f:
    payload = {
        'config': run_config_gru,
        'results': res_gru,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_gru,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)


NameError: name 'run_config_gru' is not defined

In [ ]:
# Densidade dos dados + soma das contribuições por output (uma célula)
import numpy as np
import matplotlib.pyplot as plt
import torch

# tenta localizar o seqkan em qualquer objeto do notebook
seqkan = None
model_obj = None

for name, obj in globals().items():
    try:
        if hasattr(obj, 'encoder_ode_x') and hasattr(obj.encoder_ode_x, 'seqkan'):
            seqkan = obj.encoder_ode_x.seqkan
            model_obj = obj
            break
        if hasattr(obj, 'seqkan') and hasattr(obj.seqkan, 'seqkan'):
            seqkan = obj.seqkan.seqkan
            model_obj = obj
            break
        if hasattr(obj, 'seqkan'):
            seqkan = obj.seqkan
            model_obj = obj
            break
    except Exception:
        continue

if seqkan is None:
    raise RuntimeError(
        'Nao encontrei seqkan em nenhum objeto. Execute a celula de treino ou atribua o modelo a uma variavel (ex: model).'
    )

# usa o layer de output
kan_out = seqkan.kan_out
layer = kan_out.act_fun[0]

# 1) Densidade dos dados no eixo X (por dimensão de entrada)
with torch.no_grad():
    try:
        x_data = x.detach().cpu()
    except Exception:
        # tenta obter do dataframe
        if 'dl' in globals():
            x_data = torch.tensor(dl.df[list(dl.df.columns[:-3])].to_numpy(dtype=np.float32))
        else:
            raise RuntimeError('Nao encontrei x nem dl.df para extrair dados.')

max_points = 5000
if x_data.numel() > 0 and x_data.shape[0] > max_points:
    idx = torch.randperm(x_data.shape[0])[:max_points]
    x_data = x_data[idx]

# garante 2D (N, C)
if x_data.ndim == 1:
    x_data = x_data.unsqueeze(1)


dims = list(range(x_data.shape[1]))
fig, axes = plt.subplots(len(dims), 1, figsize=(6, 2.2*len(dims)), squeeze=False)
for i, d in enumerate(dims):
    vals = x_data[:, d].numpy()
    vals = vals[~np.isnan(vals)]
    axes[i, 0].hist(vals, bins=60, density=True, alpha=0.7)
    axes[i, 0].set_title(f'Densidade X - dim {d}')
    axes[i, 0].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2) Soma das contribuições por output
from seqkan.kan.spline import coef2curve

grid = layer.grid
k = layer.k

# garante mesmo device
device = grid.device
coef = layer.coef.to(device)

x_min = grid[0, k].item()
x_max = grid[0, -k-1].item()

x_axis = torch.linspace(x_min, x_max, 200, device=device)
# avalia todas as entradas no mesmo eixo
x_eval = torch.zeros(200, layer.in_dim, device=device)
x_eval[:] = x_axis.unsqueeze(1)

y_all = coef2curve(x_eval, grid, coef, k).detach().cpu()  # (N, in_dim, out_dim)
y_sum = y_all.sum(dim=1)  # (N, out_dim)

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
for out_idx in range(y_sum.shape[1]):
    ax.plot(x_axis.detach().cpu().numpy(), y_sum[:, out_idx].numpy(), label=f'out{out_idx}')
ax.set_title('Soma das contribuições por output')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


### SeqKAN sequencial (H=24) + máscara futura + split treino/val/test\n

In [8]:
import json
from pathlib import Path
from industrial_ts.seqKAN import SeqKANSeq
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader


In [9]:
# Config proposta (H=24)
hidden_dim = 24
lag = 40
horizon = 4
feature_cols = list(dl.df.columns[:-3])
y_cols = [
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActShaft Power',
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActPress Ratio',
]
y_idx = [feature_cols.index(c) for c in y_cols]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

kan_params_seq = {
    'cell': {'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'output': {'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'k_x': 4,
        'k_h': 6,
        'k_out': 8,
        'warmup_epochs': 10,
        'mode': 'hard',
        'temp': 0.1,
    },
}

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

topk = kan_params_seq['topk']
run_name = _safe(
    f'seqKAN_seqH{hidden_dim}'
    f'_lag{lag}_h{horizon}'
    f'_kx{topk.get('k_x')}_kh{topk.get('k_h')}_kout{topk.get('k_out')}'
    f'_gcell{kan_params_seq['cell']['grid']}_kcell{kan_params_seq['cell']['k']}'
    f'_gout{kan_params_seq['output']['grid']}_kout{kan_params_seq['output']['k']}'
)
out_json = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f'{run_name}.json'


In [10]:
# Monta sequências (40 passos passados + 4 futuros) com máscara futura
# Sanitiza NaN/Inf antes de montar as sequencias
df_feat = dl.df[feature_cols].apply(pd.to_numeric, errors='coerce')
df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
df_feat = df_feat.interpolate(limit_direction='both')
df_feat = df_feat.fillna(df_feat.median())
dl.df[feature_cols] = df_feat

vals = dl.df[feature_cols].to_numpy(dtype=float)
vals = np.clip(vals, -3.0, 3.0)
T, C = vals.shape
xs, ms, ys, groups = [], [], [], []
for t in range(lag - 1, T - horizon):
    seq = vals[t - lag + 1 : t + 1 + horizon].copy()  # (44, 11)
    mask = np.ones_like(seq)
    # futuro: so 4 y observaveis
    mask[-horizon:, :] = 0.0
    mask[-horizon:, y_idx] = 1.0
    seq[-horizon:, :] *= mask[-horizon:, :]
    xs.append(seq)
    ms.append(mask)
    ys.append(vals[t + 1 : t + 1 + horizon, y_idx])
    if 'states' in dl.df.columns:
        groups.append(int(dl.df['states'].iloc[t]))

x_seq = torch.tensor(np.stack(xs), dtype=torch.float32)
m_seq = torch.tensor(np.stack(ms), dtype=torch.float32)
y_future = torch.tensor(np.stack(ys), dtype=torch.float32)
groups = np.asarray(groups) if groups else None
print(x_seq.shape, m_seq.shape, y_future.shape)


torch.Size([269787, 44, 11]) torch.Size([269787, 44, 11]) torch.Size([269787, 4, 4])


In [11]:
# Split treino/val/test (proporcional por grupo se existir)
N = x_seq.shape[0]
indices = np.arange(N)
if groups is not None:
    def _split(groups, train_frac=0.6, val_frac=0.2, seed=42):
        rng = np.random.default_rng(seed)
        tr, va, te = [], [], []
        for g in np.unique(groups):
            g_idx = indices[groups == g]
            rng.shuffle(g_idx)
            n = len(g_idx)
            n_tr = int(round(n * train_frac))
            n_va = int(round(n * val_frac))
            tr.append(g_idx[:n_tr])
            va.append(g_idx[n_tr:n_tr+n_va])
            te.append(g_idx[n_tr+n_va:])
        return np.concatenate(tr), np.concatenate(va), np.concatenate(te)
    train_idx, val_idx, test_idx = _split(groups)
else:
    rng = np.random.default_rng(42)
    rng.shuffle(indices)
    n_tr = int(0.6 * N)
    n_va = int(0.2 * N)
    train_idx, val_idx, test_idx = indices[:n_tr], indices[n_tr:n_tr+n_va], indices[n_tr+n_va:]
print('train/val/test:', len(train_idx), len(val_idx), len(test_idx))


train/val/test: 161872 53957 53958


In [12]:
# Datasets e loaders
train_ds = TensorDataset(x_seq[train_idx], m_seq[train_idx], y_future[train_idx])
val_ds = TensorDataset(x_seq[val_idx], m_seq[val_idx], y_future[val_idx])
test_ds = TensorDataset(x_seq[test_idx], m_seq[test_idx], y_future[test_idx])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)


In [13]:
# Modelo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seq_model = SeqKANSeq(
    input_size=11,
    hidden_size=hidden_dim,
    output_size=4,
    kan_params=kan_params_seq,
).to(device)

optimizer = torch.optim.Adam(seq_model.parameters(), lr=2e-4)


In [14]:
# Treino + validacao + teste + metricas (celula unica)
torch.autograd.set_detect_anomaly(True)  # opcional: ajuda a localizar NaNs
log_every = 100  # passos entre logs

def _mse_sum(pred, target):
    err = (pred - target) ** 2
    return err.sum(), err.numel()

def eval_loader(loader):
    seq_model.eval()
    losses = []
    horizon_losses = []
    with torch.no_grad():
        for xb, mb, yf in loader:
            xb = xb.to(device)
            mb = mb.to(device)
            yf = yf.to(device)
            pred = seq_model(xb, mask=mb)
            if not torch.isfinite(pred).all():
                if first_bad:
                    print('NaN/Inf no pred')
                    print(f"pred min/max: {pred.min().item():.6g} / {pred.max().item():.6g}")
                    first_bad = False
                raise RuntimeError("NaN/Inf encontrado em pred")
            pred_future = pred[:, -horizon:, :]
            sse, nobs = _mse_sum(pred_future, yf)
            loss = sse / max(nobs, 1)
            if not torch.isfinite(loss):
                if first_bad:
                    print('NaN/Inf no loss')
                    print(f"pred_future min/max: {pred_future.min().item():.6g} / {pred_future.max().item():.6g}")
                    print(f"yf min/max: {yf.min().item():.6g} / {yf.max().item():.6g}")
                    first_bad = False
                raise RuntimeError("NaN/Inf encontrado no loss")
            losses.append((float(sse), int(nobs)))
            step_mse = ((pred_future - yf) ** 2).mean(dim=(0, 2)).detach().cpu().numpy()
            horizon_losses.append(step_mse)
    h = np.stack(horizon_losses).mean(axis=0) if horizon_losses else None
    if losses:
        sse = sum(x[0] for x in losses)
        nobs = sum(x[1] for x in losses)
        return float(sse / max(nobs, 1)), h
    return float('nan'), h
epochs = 80
patience = 12
best_val = float('inf')
best_epoch = None
wait = patience
history = []

for ep in range(1, epochs + 1):
    first_bad = True
    seq_model.train()
    losses = []
    for step, (xb, mb, yf) in enumerate(train_loader, 1):
        if step == 1:
            print(f"Epoch {ep}/{epochs} | step 1/{len(train_loader)} | train (iniciando)")
        # checagem de finitude nos dados
        if not (torch.isfinite(xb).all() and torch.isfinite(mb).all() and torch.isfinite(yf).all()):
            if first_bad:
                print('NaN/Inf nos dados do batch')
                print(f"xb min/max: {xb.min().item():.6g} / {xb.max().item():.6g}")
                print(f"mb min/max: {mb.min().item():.6g} / {mb.max().item():.6g}")
                print(f"yf min/max: {yf.min().item():.6g} / {yf.max().item():.6g}")
                first_bad = False
            raise RuntimeError("NaN/Inf encontrado nos dados")
        xb = xb.to(device)
        mb = mb.to(device)
        yf = yf.to(device)
        pred = seq_model(xb, mask=mb)
        if not torch.isfinite(pred).all():
            if first_bad:
                print('NaN/Inf no pred')
                print(f"pred min/max: {pred.min().item():.6g} / {pred.max().item():.6g}")
                first_bad = False
            raise RuntimeError("NaN/Inf encontrado em pred")
        pred_future = pred[:, -horizon:, :]
        if not torch.isfinite(loss):
            if first_bad:
                print('NaN/Inf no loss')
                print(f"pred_future min/max: {pred_future.min().item():.6g} / {pred_future.max().item():.6g}")
                print(f"yf min/max: {yf.min().item():.6g} / {yf.max().item():.6g}")
                first_bad = False
            raise RuntimeError("NaN/Inf encontrado no loss")
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(seq_model.parameters(), max_norm=1.0)  # opcional
        optimizer.step()
        losses.append(loss.item())
        if step % log_every == 0:
            mean_loss = float(np.mean(losses))
            print(f'Epoch {ep}/{epochs} | step {step}/{len(train_loader)} | train {mean_loss:.6f}')
    train_loss = float(np.mean(losses))
    val_loss, val_h = eval_loader(val_loader)
    history.append({'epoch': ep, 'train_loss': train_loss, 'val_loss': val_loss, 'val_h': val_h.tolist() if val_h is not None else None})
    print(f'Epoch {ep}/{epochs} | train {train_loss:.6f} | val {val_loss:.6f} | val_h {val_h}')
    if val_loss < best_val:
        best_val = val_loss
        best_epoch = ep
        wait = patience
        torch.save(seq_model.state_dict(), Path(out_json).with_suffix('.pt'))
    else:
        wait -= 1
        if wait <= 0:
            print(f'Early stopping at epoch {ep} (best val {best_val:.6f} @ {best_epoch})')
            break

def group_metrics(mse_per_sample, groups):
    groups = np.asarray(groups)
    per_group = {}
    per_group_se = {}
    for g in np.unique(groups):
        vals = mse_per_sample[groups == g]
        per_group[int(g)] = float(np.mean(vals))
        if len(vals) >= 2:
            per_group_se[int(g)] = float(np.std(vals, ddof=1) / np.sqrt(len(vals)))
        else:
            per_group_se[int(g)] = float('nan')
    macro = float(np.mean(list(per_group.values()))) if per_group else float('nan')
    micro = float(np.mean(mse_per_sample)) if len(mse_per_sample) else float('nan')
    return macro, micro, per_group, per_group_se

# calcula mse por amostra no teste
seq_model.eval()
mse_samples = []
with torch.no_grad():
    for xb, mb, yf in test_loader:
        xb = xb.to(device)
        mb = mb.to(device)
        yf = yf.to(device)
        pred = seq_model(xb, mask=mb)
        pred_future = pred[:, -horizon:, :]
        mse = ((pred_future - yf) ** 2).mean(dim=(1, 2)).detach().cpu().numpy()
        mse_samples.append(mse)
mse_samples = np.concatenate(mse_samples) if mse_samples else np.array([])

if groups is not None:
    macro, micro, per_group, per_group_se = group_metrics(mse_samples, groups[test_idx])
    print('macro:', macro, 'micro:', micro)
    print('per_group:', per_group)
    print('per_group_se:', per_group_se)
    payload['test_macro_mse'] = macro
    payload['test_micro_mse'] = micro
    payload['test_per_group_mse'] = per_group
    payload['test_per_group_se'] = per_group_se
else:
    payload['test_micro_mse'] = float(np.mean(mse_samples)) if len(mse_samples) else float('nan')

test_loss, test_h = eval_loader(test_loader)
print('TEST loss:', test_loss, 'TEST horizon mse:', test_h)

payload = {
    'run_name': run_name,
    'config': {
        'lag': lag,
        'horizon': horizon,
        'y_cols': y_cols,
        'kan_params': kan_params_seq,
        'optimizer': 'adam',
        'lr': 2e-4,
        'batch_size': 256,
    },
    'best_val': best_val,
    'best_epoch': best_epoch,
    'test_loss': test_loss,
    'test_horizon_mse': test_h.tolist() if test_h is not None else None,
    'history': history,
}
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
print('saved:', out_json)


Epoch 1/80 | step 1/633 | train (iniciando)
Epoch 1/80 | step 100/633 | train 7.524857
Epoch 1/80 | step 200/633 | train 5.581167
Epoch 1/80 | step 300/633 | train 4.163232
Epoch 1/80 | step 400/633 | train 3.170226
Epoch 1/80 | step 500/633 | train 2.536861
Epoch 1/80 | step 600/633 | train 2.114297
Epoch 1/80 | train 2.004151 | val 0.000852 | val_h [0.00095664 0.00084009 0.000821   0.00079131]
Epoch 2/80 | step 1/633 | train (iniciando)
Epoch 2/80 | step 100/633 | train 0.001177


KeyboardInterrupt: 

### Avaliação extra: métricas por grupo, checkpoints, gráfico por passo


In [ ]:
# Salvar checkpoint last
last_ckpt = Path(out_json).with_name(f'{run_name}_last.pt')
torch.save(seq_model.state_dict(), last_ckpt)
print('saved last:', last_ckpt)


In [ ]:
# Gráfico do erro por passo (t+1..t+4)
import matplotlib.pyplot as plt
if test_h is not None:
    plt.figure(figsize=(6,4))
    plt.plot(np.arange(1, horizon+1), test_h, marker='o')
    plt.title('MSE por passo (horizonte)')
    plt.xlabel('Passo')
    plt.ylabel('MSE')
    plt.grid(True)
    plt.show()


In [ ]:
# Atualiza JSON com métricas extras
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
print('updated json:', out_json)


# Previsão 

In [8]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft', ## ou hard
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [9]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=False,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        lr_scheduler='None',
        plateau_monitor='val_micro',
        plateau_factor=0.5,
        plateau_patience=8,
        plateau_min_lr=1e-6,
        spline_soft_freeze_epoch=None,
        spline_soft_freeze_factor=0.02,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
        save_best_ckpt=True,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_predict_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    lr_scheduler=run_config['train_params']['lr_scheduler'],
    plateau_monitor=run_config['train_params']['plateau_monitor'],
    plateau_factor=run_config['train_params']['plateau_factor'],
    plateau_patience=run_config['train_params']['plateau_patience'],
    plateau_min_lr=run_config['train_params']['plateau_min_lr'],
    spline_soft_freeze_epoch=run_config['train_params']['spline_soft_freeze_epoch'],
    spline_soft_freeze_factor=run_config['train_params']['spline_soft_freeze_factor'],
    save_best_ckpt=run_config['train_params']['save_best_ckpt'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)


# JSON safe converter (numpy -> python)
def _json_safe(o):
    import numpy as _np
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    if isinstance(o, _np.ndarray):
        return o.tolist()
    if isinstance(o, _np.floating):
        return float(o)
    if isinstance(o, _np.integer):
        return int(o)
    return o

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(_json_safe(payload), f)


/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:761: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 14608, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 528}
GRUPOS (test):  {0: 4869, 1: 527}
>% |x| > 2 : 4.16%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.         -0.          2.34229483]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 1/150 | Train(sampled) L1:0.892144 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.857911 ± 0.354697 | Val micro:0.574008 ± 0.009880 
          >> Test macro:0.856798 ± 0.349457 | micro:0.576915 ± 0.009512
>% |x| > 2 : 4.12%
>% |x| > 3 : 0.00%
>% |x| > 4 : 0.00%
min: -3.0 max: 3.0
p1/p50/p99: [-3.          0.          2.29874479]
worst_min_feature: PH (CBM) 1st Stage ExpPress Ratio -3.0
worst_max_feature: PH (CBM) 1st Stage Poly Head Dev 3.0
Epoch 2/150 | Train(sampled) L1:0.888806 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.849917 ± 0.351957 | Val m

In [10]:
train_end = time.time()
train_seconds = train_end - train_start
print(f'Tempo total de treino: {train_seconds/60:.2f} min ({train_seconds:.1f} s)')

# Melhor época (menor val_micro, fallback para val_macro)
def _best_epoch(res, key_primary="val_micro", key_fallback="val_macro"):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    key = key_primary if key_primary in metrics[0] else key_fallback
    best = min(metrics, key=lambda m: m.get(key, float("inf")))
    return best.get("epoch"), best.get(key)

best_epoch, best_val = _best_epoch(res)
if best_epoch is not None:
    print(f"Melhor época: {best_epoch} | melhor val: {best_val:.6f}")
else:
    print("Não foi possível determinar a melhor época.")








Tempo total de treino: 14.82 min (889.4 s)
Melhor época: 123 | melhor val: 0.194056


# Previsão com seqKANseq

In [8]:
import importlib
import industrial_ts.seqKAN as sk
importlib.reload(sk)
from industrial_ts.seqKAN import TS_seqKANSeq


In [9]:
# checar NaN/Inf nos dados
import numpy as np
import torch

X = dl.df[list(dl.df.columns[:-3])].to_numpy(dtype=float)
print("X finite:", np.isfinite(X).all())
print("X min/max:", np.nanmin(X), np.nanmax(X))


X finite: False
X min/max: -742.3485040586738 13490.5939278987


In [10]:
import numpy as np

feature_cols = list(dl.df.columns[:-3])
df_feat = dl.df[feature_cols].apply(pd.to_numeric, errors='coerce')
df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
df_feat = df_feat.interpolate(limit_direction='both')
df_feat = df_feat.fillna(df_feat.median())

dl.df[feature_cols] = df_feat

# clamp no mesmo range do treinoS
dl.df[feature_cols] = dl.df[feature_cols].clip(-3.0, 3.0)


In [12]:
# Treino via train_cognite com TS_seqKANSeq (pipeline antigo, sem z_state)
from industrial_ts.seqKAN import TS_seqKANSeq
import time
import json as _json
from pathlib import Path

kan_params_seq = {
    'cell': {'grid': 5, 'k': 3, 'grid_range': (-3, 3)},
    'output': {'grid': 5, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'k_x': 4,
        'k_h': 6,
        'k_out': 4,
        'warmup_epochs': 30,
        'mode': 'soft',
        'temp': 0.5,
    },
}

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TS_seqKANSeq',
    in_channels=11,
    hidden_dim=24,
    output_dim=4,
    kan_params=kan_params_seq,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=80,
        validate=True,
        patience=10,
        train_fraction=train_fraction,
        horizon=1,
        y_cols=[
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActShaft Power',
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActPress Ratio',
        ],
        rebuild=False,
        reconstruction_test=False,
        optimizer_name='radam',
        optimizer_params={'lr': 2e-4},
        use_robust_scaler=True,
        x_min=-3.0,
        x_max=3.0,
        log_every=10,
    ),
)

# Nome do arquivo baseado na configuracao (padrao antigo)
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_c = kp.get('cell', {}).get('grid', 'g?')
    grid_eps = kp.get('cell', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    h = cfg.get('hidden_dim', 'H?')
    return _safe(f"seqKANseq_H{h}_g{grid_c}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

run_name = _make_run_name_seqkanseq(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_forecast_{run_name}.json")

model_seq = TS_seqKANSeq(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    output_dim=run_config['output_dim'],
    kan_params=run_config['kan_params'],
)

train_start = time.time()
res = model_seq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    train_fraction=run_config['train_params']['train_fraction'],
    horizon=run_config['train_params']['horizon'],
    y_cols=run_config['train_params']['y_cols'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    use_robust_scaler=run_config['train_params']['use_robust_scaler'],
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
    log_every=run_config['train_params']['log_every'],
)
train_end = time.time()
train_seconds = train_end - train_start

res = [r for r in res if r is not None]

# Best epoch by test micro

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

# Resumo para salvar no JSON (mesmo padrao do antigo)
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

summary = _extract_summary(res)


# JSON safe converter (numpy -> python)
def _json_safe(o):
    import numpy as _np
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    if isinstance(o, _np.ndarray):
        return o.tolist()
    if isinstance(o, _np.floating):
        return float(o)
    if isinstance(o, _np.integer):
        return int(o)
    return o

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(_json_safe(payload), f)

print(res[-1] if res else 'no results')
print('saved:', out_json)


GRUPOS (total): {0: 24345, 1: 2637}
GRUPOS (train): {0: 14607, 1: 1582}
GRUPOS (valid): {0: 4869, 1: 527}
GRUPOS (test):  {0: 4869, 1: 528}
train/val/test batches: 32/11/11
Epoch 1/80 | step 10/32 | train 0.019842
Epoch 1/80 | step 20/32 | train 0.012882
Epoch 1/80 | step 30/32 | train 0.012193
Epoch 1/80 | Train(sampled) L1:0.011584 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.000489 ± 0.000429 | Val micro:0.000834 ± 0.000547
          >> Test macro:0.004873 ± 0.003098 | micro:0.002381 ± 0.000947
          >> per_group (weighted SE): {0: '0.001775 ± 0.000857 (n=4869)', 1: '0.007971 ± 0.005587 (n=528)'}
Epoch 2/80 | step 10/32 | train 0.003635
Epoch 2/80 | step 20/32 | train 0.010558
Epoch 2/80 | step 30/32 | train 0.009954
Epoch 2/80 | Train(sampled) L1:0.011103 L2:0.000000 L3:0.000000  L4:0.000000 L5:0.000000 L6:0.000000 | Val macro:0.000433 ± 0.000433 | Val micro:0.000781 ± 0.000552
          >> Test macro:0.004856 ± 0.003125 | micro:0.002342 ± 0.000955

In [13]:
# Diagnóstico: escala/variância e baseline de persistência
import numpy as np
from sklearn.preprocessing import RobustScaler

feature_cols = list(dl.df.columns[:-3])
y_cols = [
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActShaft Power',
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActPress Ratio',
]
y_idx = [feature_cols.index(c) for c in y_cols]

# mesmos hiperparâmetros
window_size = 10
window_step = 10
horizon = 4
train_fraction = 0.6

# prepara dados iguais ao train_cognite
vals = dl.df[feature_cols].to_numpy(dtype=float)
vals[~np.isfinite(vals)] = np.nan
df_feat = dl.df[feature_cols].apply(pd.to_numeric, errors='coerce')
df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
df_feat = df_feat.interpolate(limit_direction='both')
df_feat = df_feat.fillna(df_feat.median())
vals = df_feat.to_numpy(dtype=float)

T, C = vals.shape
xs, ys = [], []
for t in range(window_size - 1, T - horizon, window_step):
    seq = vals[t - window_size + 1 : t + 1 + horizon].copy()
    xs.append(seq)
    ys.append(vals[t + 1 : t + 1 + horizon, y_idx])
x_seq = np.stack(xs)
y_future = np.stack(ys)

N = x_seq.shape[0]
idx = np.arange(N)
n_tr = int(train_fraction * N)
train_idx = idx[:n_tr]

# RobustScaler igual treino
scaler_x = RobustScaler()
scaler_x.fit(x_seq[train_idx].reshape(-1, x_seq.shape[-1]))
x_seq = scaler_x.transform(x_seq.reshape(-1, x_seq.shape[-1])).reshape(x_seq.shape)

scaler_y = RobustScaler()
scaler_y.fit(y_future[train_idx].reshape(-1, y_future.shape[-1]))
y_future = scaler_y.transform(y_future.reshape(-1, y_future.shape[-1])).reshape(y_future.shape)

# clamp igual treino
x_seq = np.clip(x_seq, -3.0, 3.0)
y_future = np.clip(y_future, -3.0, 3.0)

# baseline persistência: prever último observado
# (usa x_seq no último timestep, nos y_cols)
y_hat = x_seq[:, -1, y_idx]          # shape (N, output_dim)
y_true_last = y_future[:, -1, :]     # shape (N, output_dim)

mse = ((y_hat - y_true_last) ** 2).mean()
print("Baseline persistência (MSE no último step):", mse)

# variância média do alvo escalado
var_y = y_true_last.var(axis=0)
print("Var(y) por coluna:", var_y, " | média:", var_y.mean())


Baseline persistência (MSE no último step): 0.0
Var(y) por coluna: [0.00932987 0.00932987 0.00932987 0.00933131]  | média: 0.00933022906477966


# previsão zstate

In [ ]:
from industrial_ts.seqKAN import TSDF_seqKAN
kan_params = {
    'hidden': {'grid_eps': 0.2, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
    'output': {'grid_eps': 0.2, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'sparse_init': True,
'topk': {
    'enabled': True,
    'k_by_target': {'h_rec': 8, 's1': 12, 'z': 8},
    'warmup_epochs': 10,
    'mode': 'soft', ## ou hard
    'temp': 0.1,
    'targets': ['s1', 'h_rec', 'z'],
},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

import time
train_start = time.time()


In [ ]:
import time
from pathlib import Path
# Resumo para salvar no JSON
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    # best by test micro (micro_mse)
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    # best by val_micro if present
    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }


# Nome do arquivo baseado na configuracao
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkan(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_h = kp.get('hidden', {}).get('grid', 'g?')
    grid_eps = kp.get('hidden', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"seqKAN_g{grid_h}_ge{grid_eps}_{clamp}_{opt}_lr{lr}")

batch_size = 512
train_fraction = 0.6

run_config = dict(
    model='TSDF_seqKAN',
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=False,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        optimizer_name='radam',
        optimizer_params={'lr': 0.0002},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
    ),
)

run_name = _make_run_name_seqkan(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_predict_{run_name}.json")


model = TSDF_seqKAN(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    direct_x=run_config['direct_x'],
    kan_params=run_config['kan_params'],
)

train_start_seqkan = time.time()
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    kl_warmup_epochs=run_config['train_params']['kl_warmup_epochs'],
    kl_start=run_config['train_params']['kl_start'],
    rebuild=run_config['train_params']['rebuild'],
    reconstruction_test=run_config['train_params']['reconstruction_test'],
    warmup_steps=run_config['train_params']['warmup_steps'],
    min_lr_factor=run_config['train_params']['min_lr_factor'],
    optimizer_name=run_config['train_params']['optimizer_name'],
    optimizer_params=run_config['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config['train_params']['debug_batch_stats'],
    debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config['train_params']['x_min'],
    x_max=run_config['train_params']['x_max'],
)
train_end_seqkan = time.time()
train_seconds = train_end_seqkan - train_start_seqkan
train_minutes = train_seconds / 60.0

res = [r for r in res if r is not None]
model.save(f"{run_name}_last.pt")
# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    # uses micro_mse as test micro
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

summary = _extract_summary(res)


# JSON safe converter (numpy -> python)
def _json_safe(o):
    import numpy as _np
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    if isinstance(o, _np.ndarray):
        return o.tolist()
    if isinstance(o, _np.floating):
        return float(o)
    if isinstance(o, _np.integer):
        return int(o)
    return o

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(_json_safe(payload), f)
